In [1]:
## cnn pipeline with raw sequence inputs
## one-hot encoded

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
from tqdm import tqdm


In [2]:
# Mapping for 20 standard amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Directory setup
os.makedirs("cnn_models", exist_ok=True)
os.makedirs("cnn_results", exist_ok=True)



In [11]:
# CNN model
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=20):
        super(ProteinCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.sigmoid(self.fc(x))
        return x.squeeze()

# Dataset class

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

        lengths = [len(seq) for seq in sequences]
        min_len = min(lengths)
        max_len = max(lengths)

        if max_len - min_len > 1:
            raise ValueError(f"Sequences vary too much in length! Found lengths: {set(lengths)}")

        self.seq_len = max_len  # allow minor difference (pad shorter sequences)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        onehot = np.zeros((20, self.seq_len), dtype=np.float32)
        for i, aa in enumerate(seq):
            if i >= self.seq_len:  # (safety) but should never happen
                break
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0

        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)



In [12]:
# Training loop
def train_eval_model(gene_name, df, max_len):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = (df["Phenotype"] == "R").astype(int).tolist()

    # dataset = ProteinDataset(sequences, labels, max_len)
    dataset = ProteinDataset(sequences, labels)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)


    model = ProteinCNN()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for epoch in range(10):
        for X, y in loader:
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            all_preds.extend(out.numpy())
            all_labels.extend(y.numpy())
    auc = roc_auc_score(all_labels, all_preds)

    torch.save(model.state_dict(), f"auc_results/cnn_results/cnn_models/{gene_name}_cnn.pt")

    result_df = pd.DataFrame({
        "Gene": [gene_name],
        "AUC": [auc]
    })
    result_df.to_csv(f"auc_results/cnn_results/{gene_name}_results.csv", index=False)
    return result_df



In [13]:
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']

In [14]:
all_results = []

for gene in tqdm(gene_list):
    print("gene name", gene)
    file_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        max_len = df["Protein_Sequence"].str.len().max()
        result = train_eval_model(gene, df, max_len)
        all_results.append(result)
print(all_results)
summary_df = pd.concat(all_results, ignore_index=True)
summary_df.to_csv("auc_results/cnn_summary_auc_scores.csv", index=False)


  0%|          | 0/15 [00:00<?, ?it/s]

gene name rpoB


  7%|▋         | 1/15 [01:09<16:19, 69.93s/it]

gene name rpsL


 13%|█▎        | 2/15 [01:15<06:58, 32.17s/it]

...

gene name ethA


 93%|█████████▎| 14/15 [05:26<00:16, 16.89s/it]

gene name ethR


100%|██████████| 15/15 [05:27<00:00, 21.86s/it]

[   Gene       AUC
0  rpoB  0.961482,    Gene       AUC
0  rpsL  0.767591,    Gene       AUC
0  tlyA  0.507298,    Gene       AUC
0  pncA  0.849412,   Gene       AUC
0  eis  0.508698,   Gene       AUC
0  gid  0.751854,    Gene       AUC
0  katG  0.899059,    Gene       AUC
0  inhA  0.535122,    Gene       AUC
0  embA  0.553904,    Gene       AUC
0  embB  0.927604,    Gene       AUC
0  embC  0.561896,    Gene       AUC
0  gyrB  0.526268,    Gene       AUC
0  gyrA  0.788226,    Gene       AUC
0  ethA  0.614084,    Gene       AUC
0  ethR  0.508325]


## per residue attribution

In [31]:
# Enable gradient tracking on input.

# Run a forward pass to get model predictions.

# Backpropagate using .backward() on the prediction score (or loss).

# Extract gradients w.r.t. input → these act as importance scores.

# Aggregate across channels (i.e., amino acids) per residue.

# Store per-residue importances across the dataset.

In [20]:
def compute_residue_importance(model, dataloader, device, save_path=None):
    model.eval()
    model.to(device)
    all_importance_scores = []
    all_labels = []
    
    for inputs, labels in tqdm(dataloader, desc="Computing Residue Importance"):
        inputs = inputs.to(device)
        inputs.requires_grad_()
        outputs = model(inputs)

        grads = []
        for i in range(len(inputs)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)

            grad = inputs.grad[i].detach().cpu().numpy()  # (20, seq_len)
            score = np.abs(grad).sum(axis=0)  # sum across channels
            grads.append(score)

        all_importance_scores.extend(grads)
        all_labels.extend(labels.numpy())

        inputs.requires_grad_(False)
        inputs.grad = None

    importance_matrix = np.array(all_importance_scores)
    labels_array = np.array(all_labels)

    if save_path:
        np.savez(save_path, importance=importance_matrix, labels=labels_array)

    return importance_matrix, labels_array


In [21]:
model = ProteinCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

for gene in tqdm(gene_list):

    model.load_state_dict(torch.load(f"auc_results/cnn_results/cnn_models/{gene}_cnn.pt"))
    # Load the CSV
    df = pd.read_csv(f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv")

    # Filter valid samples
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = df["Phenotype"].map({"S": 0, "R": 1}).tolist()

    # Get max_len per gene
    max_len = max(len(seq) for seq in sequences)

    # Create Dataset + DataLoader
    dataset = ProteinDataset(sequences, labels)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
    
    importance_scores, labels = compute_residue_importance(
        model, dataloader, device,
        save_path=f"residue_importance/cnn/npz_files/{gene}_cnn_residue_importance.npz"
    )
    # Compute mean over sequences (axis=0)
    global_importance = importance_scores.mean(axis=0)

    # Save mean score as CSV
    df = pd.DataFrame({
        "Residue_Position": list(range(len(global_importance))),
        "Importance": global_importance
    })
    df.to_csv(f"residue_importance/cnn/{gene}_cnn_residue_importance_global.csv", index=False)




In [29]:
import pandas as pd
from pathlib import Path

# -------------------------------
# Constants and setup
# -------------------------------
GENE_LIST = ['rpoB', 'rpsL', 'tlyA', 'pncA', 'eis', 'gid', 'katG', 'inhA', 'embA', 'embB', 'embC', 'gyrB', 'gyrA', 'ethA', 'ethR']
K_VALUES = [10, 20, 30, 50, 100]
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
IMPORTANCE_DIR = Path("residue_importance/cnn")
CATALOG_PATH = Path("./data/filtered_variants_output.csv")
OUTPUT_PATH = Path("residue_importance/cnn_pr_overlap_summary_intersectional_only.csv")

# -------------------------------
# Functions
# -------------------------------
def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog

def compute_overlap(catalog_df, topk_df):
    important_positions = set(topk_df["Residue_Position"])
    overlap = catalog_df[catalog_df["aa_pos_0idx"].isin(important_positions)]
    overlap_sorted = overlap.merge(topk_df, left_on="aa_pos_0idx", right_on="Residue_Position")
    overlap_sorted = overlap_sorted.sort_values("Importance", ascending=False)
    tp_positions = overlap_sorted["aa_pos_0idx"].unique()
    tp = len(tp_positions)
    identified_variants = ", ".join(overlap_sorted.drop_duplicates("aa_pos_0idx")["variant"])
    return tp, identified_variants

def evaluate_gene(gene_name, catalog_df):
    results = []
    
    # Load importance scores
    imp_path = IMPORTANCE_DIR / f"{gene_name}_cnn_residue_importance_global.csv"
    if not imp_path.exists():
        print(f"Skipping {gene_name}: missing importance file.")
        return results

    imp_df = pd.read_csv(imp_path)

    # Subset WHO catalog to current gene
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if gene_catalog.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return results

    total_positives = len(set(gene_catalog["aa_pos_0idx"]))

    for k in K_VALUES:
        topk_df = imp_df.nlargest(k, "Importance")
        tp, identified_variants = compute_overlap(gene_catalog, topk_df)

        precision = tp / k if k > 0 else 0
        recall = tp / total_positives if total_positives > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall + 1e-8)

        results.append({
            "Gene": gene_name,
            "Total_Resistance_Positions": total_positives,
            "Percentile": k,
            "True_Positives": tp,
            "Precision": precision,
            "Recall": recall,
            "F1_Score": f1,
            "Identified_Variants": identified_variants
        })
    
    return results

# -------------------------------
# Main script
# -------------------------------

catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)
all_summaries = []

for gene_name in GENE_LIST:
    gene_results = evaluate_gene(gene_name, catalog_df)
    all_summaries.extend(gene_results)

final_df = pd.DataFrame(all_summaries)
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved summary to {OUTPUT_PATH}")


Skipping eis: no intersectional variants found.
Skipping embA: no intersectional variants found.
Skipping embC: no intersectional variants found.
Skipping ethR: no intersectional variants found.
Saved summary to residue_importance/cnn_pr_overlap_summary_intersectional_only.csv


In [34]:
final_df.columns

Index(['Gene', 'Total_Resistance_Positions', 'Percentile', 'True_Positives',
       'Precision', 'Recall', 'F1_Score', 'Identified_Variants'],
      dtype='object')

In [36]:
# Sort by 'gene' ascending and 'precision' descending
df_sorted = final_df.sort_values(by=['Gene', 'Precision'], ascending=[True, False])

# Now, for each gene, pick the first row (highest precision per gene)
best_per_gene = df_sorted.drop_duplicates(subset=['Gene'], keep='first').reset_index(drop=True)

print(best_per_gene)

    Gene  Total_Resistance_Positions  Percentile  True_Positives  Precision  \
0   embB                           6          10               3       0.30   
1   ethA                           9         100               3       0.03   
2    gid                           8          20               2       0.10   
3   gyrA                           5          10               3       0.30   
4   gyrB                           5          10               0       0.00   
5   inhA                           1          10               1       0.10   
6   katG                           2          20               1       0.05   
7   pncA                          95          10               9       0.90   
8   rpoB                          26          10               7       0.70   
9   rpsL                           2          10               2       0.20   
10  tlyA                           2         100               1       0.01   

      Recall  F1_Score                             

In [37]:
best_per_gene.to_csv("residue_importance/cnn_pr_overlap_summary.csv", index=False)